# 05.08 - GAN image generation

**Notebook type:** Practice notebook with theory, exercises, TODO cells, and test cases.

**Daily output:** Compact GAN training notebook and fixed-noise progression.

Build the generator–discriminator boundary, alternate updates correctly, and compare samples from fixed latent vectors before and after a short toy training run.

## Core Ideas

The generator maps latent noise to samples; the discriminator returns real/fake logits. During the discriminator update, fake samples must be detached. During the generator update, gradients must flow through discriminator operations back to the generator. Fixed noise reveals progress more reliably than changing the sample inputs every epoch.

In [ ]:
import numpy as np
import torch
from torch import nn

SEED = 5
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LATENT_DIM = 4

## Prepared Real Distribution

The real observations are noisy points on two compact arcs. This replaces an external image download while preserving adversarial update mechanics.

In [ ]:
angles = torch.linspace(0.0, float(np.pi), 64)
real_samples = torch.stack([torch.cos(angles), torch.sin(angles)], dim=1)
real_samples[32:] *= -1.0
real_samples += 0.03 * torch.randn_like(real_samples)
fixed_noise = torch.randn(12, LATENT_DIM)
print("real/fixed noise:", real_samples.shape, fixed_noise.shape)

## Exercise 05-A: Build generator and discriminator

Return raw discriminator logits for use with `BCEWithLogitsLoss`.

**Return structure — `build_gan_models`:** A dictionary with `generator` and `discriminator` `nn.Module` instances on `device`. Mappings are `[N,4]→[N,2]` and `[N,2]→[N,1]`.

In [ ]:
# TODO 05-A
def build_gan_models(latent_dim=LATENT_DIM, device=DEVICE):
    raise NotImplementedError("Complete Exercise 05-A")


# Smoke check: verify both model boundaries.
gan_models = build_gan_models()
print(gan_models["generator"](fixed_noise.to(DEVICE)).shape, gan_models["discriminator"](real_samples[:4].to(DEVICE)).shape)

## Exercise 05-B: Alternate adversarial updates

Run equal numbers of discriminator and generator updates on the complete prepared distribution.

**Return structure — `train_toy_gan`:** A `list[dict]` of length `steps`; every row contains integer `step` and Python floats `d_loss` and `g_loss`. The supplied modules are updated.

In [ ]:
# TODO 05-B
def train_toy_gan(models, real_data, steps=12, device=DEVICE):
    raise NotImplementedError("Complete Exercise 05-B")


# Smoke check and complete-data training evidence.
gan_history = train_toy_gan(gan_models, real_samples)
print("first/last:", gan_history[0], gan_history[-1])

## Exercise 05-C: Generate fixed-noise evidence

Use evaluation mode and no gradients so checkpoint comparisons are deterministic.

**Return structure — `generate_fixed_samples`:** A detached CPU `torch.float32` tensor `[N,2]`, where `N` is the number of fixed latent vectors.

In [ ]:
# TODO 05-C
def generate_fixed_samples(generator, latent_codes):
    raise NotImplementedError("Complete Exercise 05-C")


# Smoke check: generate twelve comparable outputs.
fixed_samples = generate_fixed_samples(gan_models["generator"], fixed_noise)
print("samples:", fixed_samples.shape, "spread:", float(torch.pdist(fixed_samples).mean()))

## Test Cases

**Return structure — `run_day05_tests`:** Returns `None`; assertions and `Day 05 tests passed` communicate success.

In [ ]:
def run_day05_tests():
    assert set(gan_models) == {"generator", "discriminator"}
    assert gan_models["generator"](torch.zeros(3, LATENT_DIM, device=DEVICE)).shape == (3, 2)
    assert gan_models["discriminator"](torch.zeros(3, 2, device=DEVICE)).shape == (3, 1)
    assert len(gan_history) == 12 and set(gan_history[0]) == {"step", "d_loss", "g_loss"}
    assert all(row["d_loss"] > 0 and row["g_loss"] > 0 for row in gan_history)
    assert fixed_samples.shape == (12, 2) and not fixed_samples.requires_grad
    print("Day 05 tests passed")


run_day05_tests()

## Day 05 Checklist

- [ ] Build raw-logit discriminator outputs.
- [ ] Detach generator samples during discriminator updates.
- [ ] Alternate one discriminator and one generator update.
- [ ] Reuse fixed latent vectors for evidence.
- [ ] Run the test cases.